# Lab 12 — Evaluation Harness
**Goal:** Build a tiny task-specific evaluator and compare exact-match, normalized match, and a judge-style metric.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
examples=[
("2 + 2","4"),
("capital of france","Paris"),
("translate cat to french","chat")
]
print(examples)


## Prediction
Why should the evaluation set be designed before looking at the model's final results?


In [ ]:
def norm(s):
    return " ".join(s.lower().strip().split())
preds=["4","paris","Chat"]
exact=[p==y for p,(_,y) in zip(preds,examples)]
normalized=[norm(p)==norm(y) for p,(_,y) in zip(preds,examples)]
print("exact:",sum(exact)/len(exact))
print("normalized:",sum(normalized)/len(normalized))


## Interpretation
Predefining evaluation reduces selection bias and makes improvements easier to attribute. It also encourages inclusion of known hard cases and failure modes.


In [ ]:
# Failure analysis table
for (q,y),p,e in zip(examples,preds,normalized):
    print({"prompt":q,"target":y,"prediction":p,"pass":e})


## Answers
A single aggregate score can hide regressions. Keep task-level breakdowns, qualitative examples, and failure buckets. Avoid contaminating evaluation data through training or prompt construction.

### Primary source
https://cs336.stanford.edu/


## Deepening: score slices, not just averages

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
import math, statistics

y_true = [1,1,1,0,1,0,1,1,0,1]
y_pred = [1,1,0,0,1,1,1,0,0,1]

accuracy = sum(a==b for a,b in zip(y_true,y_pred))/len(y_true)
print("accuracy:", accuracy)

# A deliberately imbalanced slice example.
slices = {
    "common": [(1,1)]*8,
    "rare": [(1,0),(0,1)]
}
for name, pairs in slices.items():
    s = sum(a==b for a,b in pairs)/len(pairs)
    print(name, "score:", s)


## Break experiment

Make the easy slice 90% of the benchmark and the hard slice 10%.

Show how the aggregate can improve while the rare-slice score gets worse.

### Required report

metric → population → slice coverage → uncertainty → release gate

Complete templates/EXPERIMENT_CARD.md for the final result.


## 7. Research exercise — build slices from real HF data


In [ ]:
!pip -q install -U datasets
from datasets import load_dataset
from collections import Counter

ds=load_dataset("CohereLabs/aya_dataset",split="train",streaming=True)
rows=[]
for row in ds:
    lang=row.get("language") or row.get("language_code")
    prompt=row.get("inputs","")
    target=row.get("targets","")
    if lang and prompt and target:
        rows.append((lang,prompt,target))
    if len(rows)>=1000: break

counts=Counter(lang for lang,_,_ in rows)
print("languages:",counts.most_common(10))
for lang in list(dict(counts.most_common(5))):
    subset=[(p,t) for l,p,t in rows if l==lang]
    print(lang,"n=",len(subset),"mean prompt chars=",round(sum(len(p) for p,t in subset)/len(subset),1))


<details><summary><strong>Expected findings / answer</strong></summary>

The aggregate sample hides language imbalance. A useful evaluation harness reports overall metrics plus language/task slices, especially when the dataset is multilingual. Do not compare slice scores without reporting sample counts and task composition. The answer is not “optimize the average”; it is to identify which slices are under-covered and whether they are release-critical.
</details>
